# Computing the Conjunctive Normal Form in First Order Logic

In order to convert a formula $f$ from first order logic into a set of clauses that is satisfiable if and only if $f$ is satisfiable,
we have to perform the following steps in order:
- eliminate biconditionals 

  Here is it is important to rename bound variables in order to avoid clashes,
- eliminate conditionals,
- transform the formula into *negation normal form*,
  i.e. we push the negation symbol inwards,

- transform the formula into *prenex normal form*,
  i.e. we move the quantifieres outside,
  
- eliminate existential quantifiers by *skolemizing* the formula, and
- transform the formula into *conjuctive normal form*,
- use set notation.

When converting formulas into conjunctive normal form, we <u>assume</u> that the formulas are 
*pure*, where we define a formula $f$ as *pure* if all quantifiers appearing in $f$ bind **different** variables.  For example, the formula
$$ \bigl(\forall X: p(X)\bigr) \vee \bigl(\forall X: q(X)\bigr)$$
is **not** *pure*, because there are two different universal quantifiers that both bind the same variable $X$.  We can rewrite this formulas as a *pure* formula by *renaming* all occurrences of $X$ that are bound by the second quantifier as follows:
$$ \bigl(\forall X: p(X)\bigr) \vee \bigl(\forall Y: q(Y)\bigr)$$

## Auxilliary Functions

Formulas are represented as trees of *records*.  In order to convert a string into a formula we use the parser that is found in the file `FOL-Parser.jsh`, which has been discussed in the notebook `FOL-Parser.ipynb` of the previous chapter.  Our parser distinguishes variables and function symbol as follows:
- A word starting with an *upper* case letter is interpreted as a *variable*.
- A word starting with a *lower* case letter is assumed to be a *function* or *predicate symbol*.

A term is either a variable `Var(x)` or a function application `App(f, args)`.  An atomic formula $p(t_1,\cdots,t_n)$ is represented as `Atom(p, args)`.  The other formulas are represented by the records `Verum`, `Falsum`, `Not`, `And`, `Or`, `Implies`, `Equiv`, `ForAll`, and `Exists`.

In [ ]:
%load ../lib/RecursiveSet.jsh
%load FOL-Parser.jsh

In [ ]:
@SafeVarargs <T> RecursiveSet<T> set(T... elements) {
    return new RecursiveSet<>(elements);
}

For testing purposes, the following formula is used.  This formula specifies the notion of a *grandparent*.

The method `toString` prints formulas in a readable form, while the function `structure` shows the records a formula is built from.

In [ ]:
var s  = "∀G:∀C:(grandparent(G, C) ↔ ∃P:(parent(G, P) ∧ parent(P, C)))";
var f1 = parse(s);
System.out.println(f1);
System.out.println(structure(f1));

A *substitution* maps variables to terms.  It is represented as an object of type `Map<String, Term>`.

The function $\texttt{applyTerm}(t, σ)$ takes a term $t$ and a *variable substitution* 
$\sigma = \{ x_1 \mapsto s_1, \cdots, x_n \mapsto s_n\}$ which is represented as a `Map` and replaces every occurrence of the variable $x_i$ in the term $t$ with the corresponding term $s_i$.

In [ ]:
// Apply the substitution sigma, which maps variables to terms, to the term t.
Term applyTerm(Term t, Map<String, Term> sigma) {
    return switch (t) {
        case Var(var x)           -> sigma.getOrDefault(x, t);
        case App(var f, var args) -> new App(f, args.stream().map(arg -> applyTerm(arg, sigma)).toList());
        default -> throw new IllegalArgumentException("unknown term: " + t);
    };
}

The function $\texttt{applyFormula}(f, σ)$ takes a formula $f$ and a *variable substitution* 
$\sigma = \{ x_1 \mapsto s_1, \cdots, x_n \mapsto s_n\}$ which is represented as a `Map` and replaces every occurrence of the variable $x_i$ in the formula $f$ with the corresponding term $s_i$.  If $\sigma$ maps a variable $x$ that is bound by a quantifier to a variable $y$, then the bound variable $x$ is renamed to $y$.

The expression `sigma.get(x) instanceof Var(var y) ? y : x` checks whether `sigma` maps `x` to a variable.  If this is the case, the name of this variable is bound to `y` and returned.  Otherwise, `x` is returned.

In [ ]:
// Apply the substitution sigma to the formula f.  If sigma maps a variable x that is
// bound by a quantifier to a variable y, then x is renamed to y.
Formula applyFormula(Formula f, Map<String, Term> sigma) {
    return switch (f) {
        case Atom(var p, var args) -> new Atom(p, args.stream().map(arg -> applyTerm(arg, sigma)).toList());
        case Verum _, Falsum _     -> f;
        case Not(var g)            -> new Not(applyFormula(g, sigma));
        case And(var g, var h)     -> new And    (applyFormula(g, sigma), applyFormula(h, sigma));
        case Or(var g, var h)      -> new Or     (applyFormula(g, sigma), applyFormula(h, sigma));
        case Implies(var g, var h) -> new Implies(applyFormula(g, sigma), applyFormula(h, sigma));
        case Equiv(var g, var h)   -> new Equiv  (applyFormula(g, sigma), applyFormula(h, sigma));
        case ForAll(var x, var g)  -> new ForAll(sigma.get(x) instanceof Var(var y) ? y : x, applyFormula(g, sigma));
        case Exists(var x, var g)  -> new Exists(sigma.get(x) instanceof Var(var y) ? y : x, applyFormula(g, sigma));
        default -> throw new IllegalArgumentException("unknown formula: " + f);
    };
}

In [ ]:
f1

In [ ]:
Map<String, Term> sigma1 = new LinkedHashMap<>();
sigma1.put("G", new Var("X"));
sigma1.put("P", new Var("Y"));
sigma1.put("C", new Var("Z"));
applyFormula(f1, sigma1)

The function $\texttt{boundVariables}(f)$ computes the set of variables that are *bound* in the formula $f$. 

In [ ]:
// The set of variables that are bound by a quantifier in the formula f.
RecursiveSet<String> boundVariables(Formula f) {
    return switch (f) {
        case Atom _, Verum _, Falsum _ -> set();
        case Not(var g)                -> boundVariables(g);
        case And(var g, var h)         -> boundVariables(g).union(boundVariables(h));
        case Or(var g, var h)          -> boundVariables(g).union(boundVariables(h));
        case Implies(var g, var h)     -> boundVariables(g).union(boundVariables(h));
        case Equiv(var g, var h)       -> boundVariables(g).union(boundVariables(h));
        case ForAll(var x, var g)      -> boundVariables(g).union(set(x));
        case Exists(var x, var g)      -> boundVariables(g).union(set(x));
        default -> throw new IllegalArgumentException("unknown formula: " + f);
    };
}

In [ ]:
f1

In [ ]:
boundVariables(f1)

The function `allVariablesTerm` computes the set of all variables that occur in the given term.

In [ ]:
// The set of all variables occurring in the term t.
RecursiveSet<String> allVariablesTerm(Term t) {
    return switch (t) {
        case Var(var x)         -> set(x);
        case App(var _, var args) -> flatMap(args, arg -> allVariablesTerm(arg));
        default -> throw new IllegalArgumentException("unknown term: " + t);
    };
}

The function `allVariables` computes the set of all variables that occur in the given formula.

In [ ]:
// The set of all variables occurring in the formula f.
RecursiveSet<String> allVariables(Formula f) {
    return switch (f) {
        case Atom(var _, var args) -> flatMap(args, arg -> allVariablesTerm(arg));
        case Verum _, Falsum _     -> set();
        case Not(var g)            -> allVariables(g);
        case And(var g, var h)     -> allVariables(g).union(allVariables(h));
        case Or(var g, var h)      -> allVariables(g).union(allVariables(h));
        case Implies(var g, var h) -> allVariables(g).union(allVariables(h));
        case Equiv(var g, var h)   -> allVariables(g).union(allVariables(h));
        case ForAll(var x, var g)  -> allVariables(g).union(set(x));
        case Exists(var x, var g)  -> allVariables(g).union(set(x));
        default -> throw new IllegalArgumentException("unknown formula: " + f);
    };
}

In [ ]:
f1

In [ ]:
allVariables(f1)

In [ ]:
Formula g1 = new Equiv(new Atom("grandparent", List.of(new Var("G"), new Var("C"))),
                       new Exists("P", new And(new Atom("parent", List.of(new Var("G"), new Var("P"))),
                                               new Atom("parent", List.of(new Var("P"), new Var("C"))))));

In [ ]:
allVariables(g1)

Below we construct a list of all upper case characters to generate new variables.

In [ ]:
// The list of all upper case letters, which are used to create new variables.
List<String> asciiUppercase = "ABCDEFGHIJKLMNOPQRSTUVWXYZ".chars().mapToObj(c -> String.valueOf((char) c)).toList();

In [ ]:
asciiUppercase

The function $\texttt{renameBoundVariables}(f)$ takes a first order formula $f$ and replaces all bound variables by **new** variables.  This only works if the list of characters `asciiUppercase` has enough characters that do not already occur in $f$.  This approach would not be good enough for a production quality program,
but for the case of a demonstration it is sufficient.  The alternative would be to rename the variables as `X1`, `X2`, `X3`, $\cdots$, but that becomes unreadable very fast.

In [ ]:
// Replace all bound variables of the formula f by new variables.
Formula renameBoundVariables(Formula f) {
    var boundVs = new ArrayList<>(boundVariables(f));
    var allVs   = allVariables(f);
    var newVars = asciiUppercase.stream().filter(x -> !allVs.contains(x)).sorted().toList();
    Map<String, Term> sigma = new LinkedHashMap<>();
    for (int i = 0; i < boundVs.size(); i++) {
        sigma.put(boundVs.get(i), new Var(newVars.get(i)));
    }
    return applyFormula(f, sigma);
}

In [ ]:
f1

In [ ]:
renameBoundVariables(f1)

## Elimination Biconditionals

The function $\texttt{eliminateBiconditional}(f)$ takes a formula $f$ from first order logic and eliminates all occurrences of the operator '↔' from this formula.  This is done by using the following equivalence:
$$(f \leftrightarrow g) \;\Leftrightarrow\; (f \rightarrow g) \wedge (g \rightarrow f)$$
In order to ensure that the resulting formula is <em style="color:blue">pure</em>, we have to rename the bound variables in the formula $g \rightarrow f$.

In [ ]:
Formula eliminateBiconditional(Formula f) {
    return switch (f) {
        case Atom _, Verum _, Falsum _ -> f;
        case Not(var g)            -> new Not(eliminateBiconditional(g));
        case And(var g, var h)     -> new And    (eliminateBiconditional(g), eliminateBiconditional(h));
        case Or(var g, var h)      -> new Or     (eliminateBiconditional(g), eliminateBiconditional(h));
        case Implies(var g, var h) -> new Implies(eliminateBiconditional(g), eliminateBiconditional(h));
        case Equiv(var g, var h)   -> {
            var ge    = eliminateBiconditional(g);
            var he    = eliminateBiconditional(h);
            var left  = new Implies(ge, he);
            var right = renameBoundVariables(new Implies(he, ge));
            yield new And(left, right);
        }
        case ForAll(var x, var g)  -> new ForAll(x, eliminateBiconditional(g));
        case Exists(var x, var g)  -> new Exists(x, eliminateBiconditional(g));
        default -> throw new IllegalArgumentException("unknown formula: " + f);
    };
}

In [ ]:
f1

In [ ]:
var f2 = eliminateBiconditional(f1);
f2

## Eliminating Conditionals

The function $\texttt{eliminateConditional}(f)$ takes a formula $f$ from first order logic and eliminates all occurrences of the operator '→' from this formula.  This is done by using the following equivalence:
$$(f \rightarrow g) \;\Leftrightarrow\; (\neg f \vee g)$$
The implementation of this function is similar to the implementation of the function `eliminateConditional` that we had used in propositional logic.

In [ ]:
Formula eliminateConditional(Formula f) {
    return switch (f) {
        case Atom _, Verum _, Falsum _ -> f;
        case Not(var g)            -> new Not(eliminateConditional(g));
        case And(var g, var h)     -> new And(eliminateConditional(g), eliminateConditional(h));
        case Or(var g, var h)      -> new Or (eliminateConditional(g), eliminateConditional(h));
        case Implies(var g, var h) -> new Or (new Not(eliminateConditional(g)), eliminateConditional(h));
        case ForAll(var x, var g)  -> new ForAll(x, eliminateConditional(g));
        case Exists(var x, var g)  -> new Exists(x, eliminateConditional(g));
        default -> throw new IllegalArgumentException("unexpected formula: " + f);
    };
}

In [ ]:
f2

In [ ]:
var f3 = eliminateConditional(f2);
f3

## Negation Normal Form

The function $\texttt{nnf}(f)$ computes the <em style="color:blue;">negation normal form</em> of $f$, while $\texttt{neg}(f)$ computes the *negation normal form* of $\neg f$. 

The auxiliary function $\texttt{neg}$ is also defined recursively:
<ol>
    <li> $\texttt{neg}(p) = \texttt{nnf}(\neg p) = \neg p$ for all propositional variables $p$,</li>
    <li> $\texttt{neg}(\neg F) = \texttt{nnf}(\neg \neg F) = \texttt{nnf}(F)$,</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \wedge F_2 \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \wedge F_2)\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \vee \neg F_2\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \vee \texttt{nnf}\bigl(\neg F_2\bigr) \\[0.1cm]
       = & \texttt{neg}(F_1) \vee \texttt{neg}(F_2).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \wedge F_2 \bigr) = \texttt{neg}(F_1) \vee \texttt{neg}(F_2)$.</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(F_1 \vee F_2 \bigr)        \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg(F_1 \vee F_2) \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1 \wedge \neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg F_1\bigr) \wedge \texttt{nnf}\bigl(\neg F_2 \bigr)  \\[0.1cm]
       = & \texttt{neg}(F_1) \wedge \texttt{neg}(F_2). 
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(F_1 \vee F_2 \bigr) = \texttt{neg}(F_1) \wedge \texttt{neg}(F_2)$.</li>
    <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(\forall x: F \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg \forall x: F\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\exists x: \neg F\bigr) \\[0.1cm]
       = & \exists x: \texttt{nnf}(\neg F)           \\[0.1cm]
       = & \exists x: \texttt{neg}(F).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(\forall x: F \bigr) = \exists x: \texttt{neg}(F)$.</li>
      <li> $$\begin{array}[t]{cl}
         & \texttt{neg}\bigl(\exists x: F \bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\neg \exists x: F\bigr) \\[0.1cm]
       = & \texttt{nnf}\bigl(\forall x: \neg F\bigr) \\[0.1cm]
       = & \forall x: \texttt{nnf}(\neg F)           \\[0.1cm]
       = & \forall x: \texttt{neg}(F).
       \end{array}
      $$
      Therefore we have $\texttt{neg}\bigl(\exists x: F \bigr) = \forall x: \texttt{neg}(F)$.</li>
</ol>

As the functions `nnf` and `neg` are mutually recursive, we define them in a single cell.

In [ ]:
Formula nnf(Formula f) {
    return switch (f) {
        case Atom _, Verum _, Falsum _ -> f;
        case Not(var g)           -> neg(g);
        case And(var g, var h)    -> new And(nnf(g), nnf(h));
        case Or(var g, var h)     -> new Or (nnf(g), nnf(h));
        case ForAll(var x, var g) -> new ForAll(x, nnf(g));
        case Exists(var x, var g) -> new Exists(x, nnf(g));
        default -> throw new IllegalArgumentException("unexpected formula: " + f);
    };
}

Formula neg(Formula f) {
    return switch (f) {
        case Verum _              -> new Falsum();
        case Falsum _             -> new Verum();
        case Not(var g)           -> nnf(g);
        case And(var g, var h)    -> new Or (neg(g), neg(h));
        case Or(var g, var h)     -> new And(neg(g), neg(h));
        case Atom _               -> new Not(f);
        case ForAll(var x, var g) -> new Exists(x, neg(g));
        case Exists(var x, var g) -> new ForAll(x, neg(g));
        default -> throw new IllegalArgumentException("unexpected formula: " + f);
    };
}

In [ ]:
f3

In [ ]:
var f4 = nnf(f3);
f4

## Prenex Normal Form

In the following we assume that all quantifiers that occur in a formula bind **different** variables, i.e. we assume that the formulas are *pure*.  If this assumption is not satisfied, then the functions given below will produce <u>garbage</u>.

A quantifier together with the variable it binds, e.g. $\forall X$, is represented as an object of the record class `Quantifier`.  The function `cons(q, rest)` returns the list that starts with `q` and continues with the elements of `rest`.

In [ ]:
// A quantifier together with the variable it binds, e.g. ∀X.
record Quantifier(String symbol, String variable) {
    public String toString() { return symbol + variable; }
}

// Returns the list consisting of q followed by the elements of rest.
List<Quantifier> cons(Quantifier q, List<Quantifier> rest) {
    List<Quantifier> result = new ArrayList<>();
    result.add(q);
    result.addAll(rest);
    return result;
}

A *quantifier list* is a list of the following form:
$$ [Q_1 x_1, \cdots, Q_n x_n] $$
Here, the $Q_i$ denote quantifiers, i.e. we have $Q_i \in \{\forall, \exists\}$, while the $x_i$ are variables.  The function $\texttt{mergeQuantifiers}(T_1, T_2)$ takes two quantifier lists $T_1$ and $T_2$ as arguments and merges them into a new quantifier list such that the relative order of the quantifiers remains the same, i.e. if both $Q_1 x_1$ and $Q_2 x_2$ occur in $T_1$ and $Q_1 x_1$ occurs before $Q_2 x_2$, then $Q_1 x_1$ will occur before $Q_2 x_2$ in the result.

In [ ]:
List<Quantifier> mergeQuantifiers(List<Quantifier> Q1, List<Quantifier> Q2) {
    if (Q1.isEmpty()) return Q2;
    if (Q2.isEmpty()) return Q1;
    if (Q1.get(0).symbol().equals("∃")) return cons(Q1.get(0), mergeQuantifiers(Q1.subList(1, Q1.size()), Q2));
    if (Q2.get(0).symbol().equals("∃")) return cons(Q2.get(0), mergeQuantifiers(Q1, Q2.subList(1, Q2.size())));
    return cons(Q1.get(0), mergeQuantifiers(Q1.subList(1, Q1.size()), Q2));
}

In [ ]:
mergeQuantifiers(List.of(new Quantifier("∀", "X"), new Quantifier("∃", "Y")),
                 List.of(new Quantifier("∃", "U"), new Quantifier("∀", "V")))

Given a formula $f$, the function $\texttt{extractQuantifiers}(f)$ returns a pair $(T, m)$, where $T$ is a quantifier list and $m$ is the <em style="color:blue;">matrix</em> of the formula $f$, where the matrix of a formula is defined as the part that remains when all quantifiers have been extracted.  The pair is an object of the record class `Pair`, which is defined in the file `RecursiveSet.jsh`.

In [ ]:
Pair<List<Quantifier>, Formula> extractQuantifiers(Formula f) {
    return switch (f) {
        case Atom _, Verum _, Falsum _, Not _ -> new Pair<>(List.of(), f);
        case And(var g, var h) -> {
            var gq = extractQuantifiers(g);
            var hq = extractQuantifiers(h);
            yield new Pair<>(mergeQuantifiers(gq.first(), hq.first()), new And(gq.second(), hq.second()));
        }
        case Or(var g, var h) -> {
            var gq = extractQuantifiers(g);
            var hq = extractQuantifiers(h);
            yield new Pair<>(mergeQuantifiers(gq.first(), hq.first()), new Or(gq.second(), hq.second()));
        }
        case ForAll(var x, var g) -> {
            var gq = extractQuantifiers(g);
            yield new Pair<>(cons(new Quantifier("∀", x), gq.first()), gq.second());
        }
        case Exists(var x, var g) -> {
            var gq = extractQuantifiers(g);
            yield new Pair<>(cons(new Quantifier("∃", x), gq.first()), gq.second());
        }
        default -> throw new IllegalArgumentException("unexpected formula: " + f);
    };
}

In [ ]:
f4

In [ ]:
var qm = extractQuantifiers(f4);
var Qs = qm.first();
var f5 = qm.second();
System.out.println(Qs);
System.out.println(f5);

Given a quantifier list $\texttt{Qs}$ and a matrix $m$, the call $\texttt{attachQuantifiers}(Qs, m)$ combines the quantifiers $\texttt{Qs}$ and the matrix $m$ into a quantified formula.

In [ ]:
Formula attachQuantifiers(List<Quantifier> Qs, Formula m) {
    if (Qs.isEmpty()) return m;
    var q    = Qs.get(0);
    var rest = attachQuantifiers(Qs.subList(1, Qs.size()), m);
    return q.symbol().equals("∀") ? new ForAll(q.variable(), rest) : new Exists(q.variable(), rest);
}

In [ ]:
Qs

In [ ]:
f5

In [ ]:
var f6 = attachQuantifiers(Qs, f5);
f6

## Skolemization (Eliminating Existential Quantifiers)

The variable $\texttt{skolemCounter}$ is a global variable that is needed to create unique Skolem constants.  

In [ ]:
int skolemCounter = 0;

In [ ]:
String skolemConstant() {
    skolemCounter += 1;
    return "sk" + skolemCounter;
}

The function $\texttt{skolemize}(f, \texttt{Vs})$ takes a formula $f$ and a list of variables $\texttt{Vs}$ and 
<em style="color:blue">skolemizes</em> the formula $f$, i.e. it replaces all existentially quantified variables by appropriate <em style="color:blue">Skolem functions</em>.  The list $\texttt{Vs}$ is a list of variables that are 
assumed to be universally quantified.  The formula $f$ is assumed to be in <em style="color:blue">prenex normal form</em>.

For skolemization to work correctly, we have to assume that 
<font size="4" style="color:darkgreen; size:125%">$f$ does not contain free variables</font>!

In [ ]:
Formula skolemize(Formula f, List<String> Vs) {
    return switch (f) {
        case Exists(var x, var g) -> {
            Term t = new App(skolemConstant(), Vs.stream().map(v -> (Term) new Var(v)).toList());
            yield skolemize(applyFormula(g, Map.of(x, t)), Vs);
        }
        case ForAll(var x, var g) -> {
            List<String> newVs = new ArrayList<>(Vs);
            newVs.add(x);
            yield new ForAll(x, skolemize(g, newVs));
        }
        default -> f;
    };
}

In [ ]:
f6

In [ ]:
var f7 = skolemize(f6, List.of());
f7

## Conversion to Clauses

A *literal* is either an atomic formula or the negation of an atomic formula.  Hence a literal is either an object of the record class `Atom` or an object of the form `Not(Atom(...))`.  As records compare their components structurally, literals can be inserted into sets without any conversion.  Therefore, a clause is a `RecursiveSet<Formula>`, and a set of clauses has the type `RecursiveSet<RecursiveSet<Formula>>`.

The function $\texttt{cnf}(f)$ takes a <em style="color:blue">skolemized</em> formula $f$ from first order logic that is in <em style="color:blue">negation normal form</em> and returns the <em style="color:blue">conjunctive normal form</em> of $f$ in <em style="color:blue">set notation</em>.  This works the same way as in propositional logic.

In [ ]:
RecursiveSet<RecursiveSet<Formula>> cnf(Formula f) {
    return switch (f) {
        case Verum _              -> set();
        case Falsum _             -> set(set());
        case Atom _, Not _        -> set(set(f));
        case And(var g, var h)    -> cnf(g).union(cnf(h));
        case Or(var g, var h)     -> flatMap(cnf(g), k1 -> cnf(h).map(k2 -> k1.union(k2)));
        case ForAll(var x, var g) -> cnf(g);
        default -> throw new IllegalArgumentException("unexpected formula: " + f);
    };
}

In [ ]:
f7

In [ ]:
var f8 = cnf(f7);
for (var cl : f8) {
    System.out.println(cl);
}

## Putting Everything Together

The function `normalize` takes a <em style="color:blue">pure</em> formula $f$ from first order logic and transforms $f$ into a set of first order clauses.  Furthermore, $f$ **must not** contain free variables.

In [ ]:
RecursiveSet<RecursiveSet<Formula>> normalize(Formula f) {
    var f1 = eliminateBiconditional(f);
    var f2 = eliminateConditional(f1);
    var f3 = nnf(f2);
    var qm = extractQuantifiers(f3);
    var f4 = attachQuantifiers(qm.first(), qm.second());
    var f5 = skolemize(f4, List.of());
    return cnf(f5);
}

In [ ]:
for (var cl : normalize(f1)) {
    System.out.println(cl);
}

In [ ]:
// Convert a set of clauses into a string that shows every clause on a separate line.
String prettify(RecursiveSet<RecursiveSet<Formula>> M) {
    if (M.isEmpty()) {
        return "{}";
    }
    return M.stream().map(C -> "    " + C).collect(Collectors.joining(",\n", "{\n", "\n}"));
}

In [ ]:
void test(String s) {
    var f = parse(s);
    System.out.println("The knf of " + s + " is:");
    System.out.println(prettify(normalize(f)));
}

In [ ]:
test(s);

In [ ]:
test("¬(∃Y:∀X:p(X,Y)→∀U:∃V:p(U,V))");

The functions defined in this notebook are also available in the file `FOL-CNF.jsh`.  This file is used in the notebook `03-Prover.ipynb`.